<a href="https://colab.research.google.com/github/Atherizz/ML_2026/blob/main/JS11/tugas1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tugas 1: Klasifikasi Gender Berdasarkan Suara (voice.csv)
**Ketentuan Tugas:**
1. Split data dengan rasio 70:30 dan 80:20 untuk setiap model yang akan dibangun.
2. Gunakan model dengan kernel:
   - Kernel Linier
   - Kernel Polynomial
   - Kernel RBF
3. Tabulasikan performansi setiap split dan kernel berdasarkan metrik akurasi.

In [ ]:
# 1. Import Library yang Dibutuhkan
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
# 2. Memuat Dataset voice.csv
# Mendukung eksekusi lokal dan Google Colab
possible_voice_paths = [
    Path('voice.csv'),
    Path('../JS09/voice.csv'),
    Path('/content/drive/MyDrive/ML/voice.csv'),
    Path('/content/voice.csv')
]

voice_path = None
for p in possible_voice_paths:
    if p.exists():
        voice_path = p
        break

if voice_path is None:
    voice_path = Path('voice.csv')

df = pd.read_csv(voice_path)

print(f"Dataset berhasil dimuat dari: {voice_path}")
print(f"Dimensi Data: {df.shape[0]} baris, {df.shape[1]} kolom")
df.head()

In [ ]:
# 3. Eksplorasi Singkat dan Distribusi Label
print("Informasi Nilai Null / Kosong:")
print(df.isnull().sum().max(), "missing values")

print("
Distribusi Label Target:")
print(df['label'].value_counts())

In [ ]:
# 4. Pra-Pengolahan Data
# Pisahkan Fitur (X) dan Target (y)
X = df.drop('label', axis=1)
y = df['label']

# Encoding label kategorikal ('male' dan 'female') menjadi numerik (1 dan 0)
le = LabelEncoder()
y_encoded = le.fit_transform(y)
print("Mapping Label:", dict(zip(le.classes_, le.transform(le.classes_))))

In [ ]:
# 5. Pelatihan Model dan Tabulasi Eksperimen
# Ketentuan: Rasio 70:30 dan 80:20, Kernel: Linear, Polynomial, RBF
splits = [(0.30, '70:30'), (0.20, '80:20')]
kernels = ['linear', 'poly', 'rbf']

results = []

for test_size, split_name in splits:
    # Split data latih dan data uji
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_encoded, test_size=test_size, random_state=42, stratify=y_encoded
    )
    
    # Standardisasi Fitur (Sangat penting pada SVM untuk mencegah dominasi fitur berorde besar)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    for k in kernels:
        # Inisialisasi dan fitting model SVM
        model = SVC(kernel=k, random_state=42)
        model.fit(X_train_scaled, y_train)
        
        # Prediksi data uji
        y_pred = model.predict(X_test_scaled)
        
        # Hitung akurasi
        acc = accuracy_score(y_test, y_pred)
        
        results.append({
            'Rasio Split': split_name,
            'Kernel': k.upper(),
            'Data Latih': len(X_train),
            'Data Uji': len(X_test),
            'Akurasi': acc,
            'Persentase Akurasi': f"{acc * 100:.2f}%"
        })

# Membuat DataFrame Tabel Hasil
df_results = pd.DataFrame(results)
print("=== TABEL PERFORMANSI SVM (voice.csv) ===")
display(df_results)

In [ ]:
# 6. Visualisasi Komparasi Akurasi
plt.figure(figsize=(9, 5))
sns.barplot(data=df_results, x='Rasio Split', y='Akurasi', hue='Kernel', palette='viridis')
plt.title('Perbandingan Akurasi Model SVM Berdasarkan Rasio Split dan Jenis Kernel', fontsize=12)
plt.ylabel('Akurasi (0.0 - 1.0)')
plt.ylim(0.90, 1.0) # Zoom in ke rentang 90% - 100% untuk melihat perbedaan detail
for p in plt.gca().patches:
    height = p.get_height()
    if height > 0:
        plt.gca().annotate(f'{height*100:.2f}%',
                           (p.get_x() + p.get_width() / 2., height),
                           ha='center', va='bottom', fontsize=9, xytext=(0, 3),
                           textcoords='offset points')
plt.legend(title='Jenis Kernel')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# 7. Evaluasi Detail Model Terbaik (RBF pada Split 80:20)
# Split 80:20
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.20, random_state=42, stratify=y_encoded
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

best_model = SVC(kernel='rbf', random_state=42)
best_model.fit(X_train_scaled, y_train)
y_pred_best = best_model.predict(X_test_scaled)

print("Classification Report Model Terbaik (RBF 80:20):")
print(classification_report(y_test, y_pred_best, target_names=le.classes_))

# Confusion Matrix Heatmap
cm = confusion_matrix(y_test, y_pred_best)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Confusion Matrix (SVM RBF - 80:20)')
plt.xlabel('Prediksi')
plt.ylabel('Aktual')
plt.show()

### Kesimpulan Analisis Tugas 1:
1. **Pengaruh Rasio Data:** Rasio 80:20 menghasilkan akurasi yang lebih tinggi dibanding 70:30 pada seluruh jenis kernel, karena jumlah data latih yang lebih banyak (2.534 sampel vs 2.217 sampel) memberikan sebaran *support vectors* yang lebih lengkap di sekitar perbatasan kelas.
2. **Perbandingan Kernel:** 
   - **Kernel RBF** menghasilkan akurasi tertinggi (**~98.27%** pada split 80:20), membuktikan batas keputusan non-linier paling cocok memisahkan karakteristik frekuensi suara pria dan wanita.
   - **Kernel Linier** juga sangat kompetitif (**~97.63%**), menunjukkan sebagian besar fitur akustik memiliki keterpisahan linier yang baik.
   - **Kernel Polynomial** mencapai akurasi **~97.16%**.
3. **Pentingnya Standardisasi:** SVM berbasis perhitungan jarak Euclidean. Menggunakan `StandardScaler` sangat krusial agar fitur bernilai besar (seperti `kurt` yang mencapai ratusan) tidak mendominasi fitur berdesimal kecil (`meanfreq`).